In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../..")

import analytiq_data as ad
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("test_textract")

In [3]:
ad.common.setup()

In [4]:
analytiq_client = ad.common.get_analytiq_client(env="test")
aws_client = await ad.aws.get_aws_client_async(analytiq_client)

In [5]:
document_id = "67aa2cae086d8b1d7826ef36"
document_id = "679c9abbfcb627f388ffd59f"
doc = await ad.common.get_doc(analytiq_client, document_id)
if doc is None:
    raise Exception("Document not found")

file_name = doc.get("pdf_file_name") or doc.get("mongo_file_name")
file_obj = await ad.common.get_file_async(analytiq_client, 
                                          file_name)
if file_obj is None:
    raise Exception("File not found")

blob = file_obj["blob"]

In [6]:
logger.info(f"blob size: {len(blob)}")
logger.info(f"blob head: {blob[:100]}")

INFO:test_textract:blob size: 114199
INFO:test_textract:blob head: b'%PDF-1.7\r\n%\xb5\xb5\xb5\xb5\r\n1 0 obj\r\n<</Type/Catalog/Pages 2 0 R/Lang(hu) /StructTreeRoot 40 0 R/MarkInfo<</Mar'


In [7]:

blocks = await ad.aws.textract.run_textract(analytiq_client, 
                                            blob, 
                                            document_id=document_id)
logger.info(f"blocks: {len(blocks)}")

INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 0: IN_PROGRESS
INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 1: IN_PROGRESS
INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 2: IN_PROGRESS
INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 3: IN_PROGRESS
INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 4: IN_PROGRESS
INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 5: IN_PROGRESS
INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 6: SUCCEEDED
INFO:analytiq_data.aws.textract:None: 679c9abbfcb627f388ffd59f: step 0: blocks len: 511 next_token: None
INFO:test_textract:blocks: 4


In [9]:
import textractor

In [10]:
document = textractor.parsers.response_parser.parse(blocks)

In [12]:
# Save the document to a markdown file
with open("document.md", "w") as f:
    f.write(document.to_markdown())

In [11]:
document.to_markdown()

'Andras Tremmel\n\n\nhttps://github.com/AndrasTremmel\n\n\nin\n\n\nwww.linkedin.com/in/andras-tremmel\n\n\ntremmel.andris@gmail.com\n\n\nEducation\n\n\nImperial College London - MEng Computing\n\n\nOct 2022 - Jun 2026\n\n\nSkills\n\n\nProgramming languages C, C++, Haskell, Kotlin, Java, Scala, Python, Prolog, Verilog, Assembly, SQL\n\n\nTools: GitHub, GitLab CI/CD, Eclipse, Vim, IntelliJ IDEA, VSCode, GDB, SupaBase\n\n\nFrameworks PyQt, Node.js, SDL, libGDX\n\n\nLanguages: Hungarian - Native, English - Fluent (CAE: C1), German - Intermediate (ÖSD: B2)\n\n\nProjects\n\n\nYourDestiny (Project with two friends of mine) I Java, libGDX\n\n\nMay 2024 - Current\n\n\nDeveloping a 2D pixel survival and exploration game following the MV architectural design pattern and\n\n\nutilising the game developing features of libGDX.\n\n\nHighly focusing on polymorphism and abstraction to make the program flexible to further extensions and\n\n\noptimisations.\n\n\nWiseBrowse I Python, PyQt, OpenAI, SupaBas